# Pokemon Platinum AI
<b> Playing Pokemon through Reinforcement Learning w/ Human Feedback </b> <br>
*Tools used:*
- PyTorch (training model)
- pyautogui (connecting to emulation)
- 


In [17]:
# Importing packages
import torch
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
import pyautogui
import subprocess
import time

In [18]:
# Setting the  environment
def run_lua_script():
    """Run DeSmuME (Emulator) with Lua script"""
    desmume_path = '/Applications/DeSmuME.app'
    lua_script_path = 'env.lua'
    subprocess.Popen([desmume_path, '--lua-script', lua_script_path])

In [19]:
# Get current frame
def read_coordinates():
    """Read coordinates from the Lua script output"""
    with open('/path/to/output.txt', 'r') as file:
        lines = file.readlines()
        last_line = lines[-1]
        coordinates = last_line.split('=')[1].strip().split(',')
        x, y = int(coordinates[0]), int(coordinates[1])
        return x, y

In [20]:
def send_key_press(key):
    pyautogui.press(key)

In [21]:
def main():
    run_lua_script()
    time.sleep(5)

    while True:
        x, y = read_coordinates()
        print(f"Player Coordinates: X={x}, Y={y}")

        # Example: Move player based on coordinates
        if x < 100:
            send_key_press('right')
        elif x > 200:
            send_key_press('left')

        time.sleep(0.1)


In [22]:
# Defining Deep Q-Network (DQN) Model
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [23]:
# Initialise our loss, model, and optimizer
loss_fn = nn.MSELoss()
model = DQN(input_dim = 4, output_dim= 2)
optimizer = optim.Adam(model.parameters())

In [24]:
# Define replay buffer to store the agent's experiences, sample them randomly during training
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return np.array(state), np.array(action), np.array(reward), np.array(next_state), np.array(done)

    def __len__(self):
        return len(self.buffer)


In [ ]:
# Applying HF (Human Feedback)
def get_human_feedback(state, action):
    # Implement a mechanism to get human feedback
    # For example, +1 for positive feedback, -1 for negative feedback, 0 for no feedback
    feedback = input("Provide feedback for action {} in state {}: ".format(action, state))
    if feedback == '+':
        return 1
    elif feedback == '-':
        return -1
    else:
        return 0

In [25]:
# Training Loop
def train_dqn(model, replay_buffer, batch_size, gamma):
    if len(replay_buffer) < batch_size:
        return

    state, action, reward, next_state, done = replay_buffer.sample(batch_size)

    state = torch.FloatTensor(state)
    action = torch.LongTensor(action)
    reward = torch.FloatTensor(reward)
    next_state = torch.FloatTensor(next_state)
    done = torch.FloatTensor(done)

    q_values = model(state)
    next_q_values = model(next_state)

    q_value = q_values.gather(1, action.unsqueeze(1)).squeeze(1)
    next_q_value = next_q_values.max(1)[0]
    expected_q_value = reward + gamma * next_q_value * (1 - done)

    loss = loss_fn(q_value, expected_q_value)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Parameters
gamma = 0.99
batch_size = 32
replay_buffer = ReplayBuffer(10000)

# Example training loop
for episode in range(1000):
    state = env.reset()
    done = False
    while not done:
        action = model(state).argmax().item()
        next_state, reward, done, _ = env.step(action)

        # Apply human feedback (positive/negative reward adjustment)
        human_feedback = get_human_feedback(state, action)
        reward += human_feedback

        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state

        train_dqn(model, replay_buffer, batch_size, gamma)


NameError: name 'env' is not defined